In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import numpy.fft as fft
from matplotlib.patches import Circle
root = Path("data2d")

criteria = {
    "status": "complete",
    "vx": 0.05,
    "coupling": 2.0,
    "field_length_scale": 1.0,
}

def matches_value(value, criterion):
    if callable(criterion):
        return bool(criterion(value))
    if isinstance(criterion, (set, frozenset)):
        return any(matches_value(value, item) for item in criterion)
    if isinstance(criterion, (float, np.floating)):
        return bool(np.isclose(value, criterion, rtol=1e-9, atol=1e-12))
    return value == criterion

matches = []

for param_file in sorted(root.rglob("params.json")):
    with param_file.open() as f:
        params = json.load(f)

    if all(
        key in params and matches_value(params[key], criterion)
        for key, criterion in criteria.items()
    ):
        matches.append({"folder": param_file.parent, "params": params})

print(f"Found {len(matches)} matches.")
for i, match in enumerate(matches):
    print(f"[{i}] {match['folder']}")

In [ ]:
index = 0  # Choose from the matches above.

selected = matches[index]
parent_folder = selected["folder"]
params = selected["params"]

phi_path = parent_folder / "phi.npy"
phi = np.load(phi_path, mmap_mode="r") if phi_path.exists() else None
phi_times = np.load(parent_folder / "phi_times.npy", mmap_mode="r")
snapshot_steps = np.load(parent_folder / "snapshot_steps.npy", mmap_mode="r")
z_hist = np.load(parent_folder / "z_hist.npy", mmap_mode="r")
times = np.load(parent_folder / "times.npy", mmap_mode="r")

print("Loaded:", parent_folder)

In [ ]:
trap_origin = np.asarray(params["trap_origin"])
v = np.array([params["vx"], params["vy"]])

In [ ]:
# Saved field resolution
dx = params.get("snapshot_dx", params["dx"])
dy = params.get("snapshot_dy", params.get("dy", params["dx"]))

Lx = params["Lx"]
Ly = params["Ly"]

Nx_phi = phi.shape[1]
Ny_phi = phi.shape[2]

trap_origin = np.asarray(
    params.get("trap_origin", [Lx / 2, Ly / 2]),
    dtype=float,
)

vx = float(params.get("vx", 0.0))
vy = float(params.get("vy", 0.0))

# Interpret r0 as the particle radius
particle_radius = float(params["r0"])

phi_low, phi_high = np.quantile(phi, [0.01, 0.99])


for phi_idx in range(0, phi.shape[0], 5):

    t = float(phi_times[phi_idx])

    hist_idx = np.clip(
        np.searchsorted(times, t),
        0,
        len(times) - 1,
    )

    # Moving trap reference in y.
    # For the present case vy = 0 and this is simply Ly/2.
    y_reference = (
        trap_origin[1] + vy * t
    ) % Ly

    # Grid index corresponding to the reference y.
    iy_reference = int(np.rint(y_reference / dy)) % Ny_phi

    # Roll only along y so the trap center is at y = 0.
    phi_y_centered = np.roll(
        phi[phi_idx],
        shift=Ny_phi // 2 - iy_reference,
        axis=1,
    )

    # phi is stored as (x, y), while imshow expects (y, x).
    phi_plot = phi_y_centered.T

    # Laboratory x-coordinate: x remains unshifted.
    particle_x = z_hist[hist_idx, 0] % Lx

    # Particle y-coordinate relative to the trap center.
    particle_y = (
        z_hist[hist_idx, 1]
        - y_reference
        + Ly / 2
    ) % Ly - Ly / 2

    fig, ax = plt.subplots(figsize=(6, 4))

    im = ax.imshow(
        phi_plot,
        origin="lower",
        extent=[0, Lx, -Ly / 2, Ly / 2],
        vmin=phi_low,
        vmax=phi_high,
        cmap="RdBu_r",
        aspect="equal",
    )

    # Draw periodic copies so the particle remains visible at x-boundaries.
    for shift_x in (-Lx, 0.0, Lx):
        for shift_y in (-Ly, 0.0, Ly):

            particle = Circle(
                (
                    particle_x + shift_x,
                    particle_y + shift_y,
                ),
                radius=particle_radius,
                facecolor="gray",
                edgecolor="black",
                linewidth=0.8,
                zorder=3,
                alpha=0.8
            )

            ax.add_patch(particle)

    ax.set_xlim(0, Lx)
    ax.set_ylim(-Ly / 2, Ly / 2)

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y-y_{\mathrm{trap}}$")
    ax.set_title(f"$t = {t:.2f}$")

    fig.colorbar(im, ax=ax, label=r"$\phi$")
    fig.tight_layout()
    plt.savefig("plots/phi_{:04d}.png".format(phi_idx), dpi=300, bbox_inches='tight')
    plt.show()